# Leverage Survival Lab — Main Results

本 notebook は実験結果(`results/grid_real_btc_n500.parquet`)から、README に掲載されている図表を再現します。

事前に `python scripts/run_realdata_experiment.py --n-windows 500 --name real_btc_n500` を実行してください。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from leverage_survival_lab.analysis.stats import survival_summary, wilson_ci

df = pd.read_parquet('../results/grid_real_btc_n500.parquet')
df = df[df['error'].isna()] if 'error' in df.columns else df
print(f'total simulations: {len(df):,}')
df.head()

## H1 — 100倍レバの30日生存率

全 (戦略, 損切) セルで生存率と Wilson 95% CI を算出。

In [ ]:
summary = survival_summary(df)
h1 = summary[summary['leverage'] == 100.0].copy()
h1.loc[:, ['strategy','stop_loss','n','survival','ci_lo','ci_hi']]

## Hero ヒートマップ — 全戦略集約

In [ ]:
agg = (
    df.assign(survives=lambda d: (d['final_equity'] >= 100_000).astype(int))
      .groupby(['leverage', 'stop_loss'], dropna=False)['survives']
      .mean().unstack()
)
agg = agg.sort_index().sort_index(axis=1, na_position='last')

fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.imshow(agg.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(agg.columns)))
ax.set_xticklabels([f'{c*100:.1f}%' if isinstance(c, float) and not np.isnan(c) else 'None' for c in agg.columns], fontsize=10)
ax.set_yticks(range(len(agg.index)))
ax.set_yticklabels([f'{int(v)}x' for v in agg.index], fontsize=10)
ax.set_xlabel('Stop Loss')
ax.set_ylabel('Leverage')
ax.set_title(f'30-day Survival Rate (N={len(df):,})')
for i in range(agg.shape[0]):
    for j in range(agg.shape[1]):
        v = agg.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v*100:.0f}%', ha='center', va='center', color='black' if v > 0.4 else 'white', fontsize=11)
fig.colorbar(im, ax=ax, label='Survival rate')
plt.tight_layout()
plt.show()

## H4 — 50x以上で戦略の差は消える(2比率検定 + Bonferroni)

In [ ]:
from leverage_survival_lab.analysis.stats import two_proportion_z, bonferroni
rows = []
for L in [50.0, 100.0]:
    rand = df[(df['leverage'] == L) & (df['strategy_name'] == 'random')]
    p_rand = float((rand['final_equity'] >= 100_000).mean())
    n_rand = len(rand)
    for s in ['sma_cross','rsi','bollinger','breakout']:
        sub = df[(df['leverage'] == L) & (df['strategy_name'] == s)]
        p = float((sub['final_equity'] >= 100_000).mean())
        z, pv = two_proportion_z(p, len(sub), p_rand, n_rand)
        rows.append({'lev': L, 'strategy': s, 'p_strategy': p, 'p_random': p_rand, 'p_value': pv})
h4 = pd.DataFrame(rows)
rejects, alpha = bonferroni(h4['p_value'].fillna(1.0).tolist())
h4['reject'] = rejects
h4['alpha_adj'] = alpha
h4

## レバ × 平均終端残高

In [ ]:
by_lev = df.groupby('leverage').agg(
    mean_final=('final_equity', 'mean'),
    median_final=('final_equity', 'median'),
    bust_rate=('is_bust', 'mean'),
    median_dd=('max_drawdown', 'median'),
    n=('final_equity', 'count'),
).round(0)
by_lev